# Telegram Movie1 uploader — no Codex installation needed

Prepared **12 November 2026** from the inspected GitHub export.

**2,573 selected posts: 2,532 photos with captions + 41 text posts.**
Excluded: 29 missing-video posts and 1,219 duplicate/superseded photo posts.

Run the numbered cells in order in a **standard CPU runtime**. This notebook contains
the cleaned export, a reversible removal ledger, and the uploader. It fetches the
selected photos from your GitHub snapshot as needed. You do not need to upload them manually.

Original/official Hindi labels are preferred, then AI Hindi if explicitly labelled,
then other Hindi; the newest saved edit/post wins within a tier. Other-language titles
are retained if no Hindi version is present. No explicit AI dub was detected in the
audio/header labels. Language claims and external links have **not** been verified.
Full selected captions, including old promotional/channel links, are preserved.

**Nothing posts until you type the destination-specific send confirmation.**
The bot token goes in a hidden input, never in source code. You do not supply your
Telegram password, phone login code, or personal Telegram session to this program.


In [ ]:
#@title 1. Prepare files and persistent progress folder
from pathlib import Path
import base64, hashlib, importlib.util, io, json, sys, zipfile
try:
    from google.colab import drive
except ImportError:
    WORKDIR = Path.cwd() / 'Telegram_Movie1_Migration'
else:
    drive.mount('/content/drive')
    WORKDIR = Path('/content/drive/MyDrive/Telegram_Movie1_Migration')
WORKDIR.mkdir(parents=True, exist_ok=True)
BUNDLE_B64 = '...'
EXPECTED_BUNDLE_SHA256 = 'c2d2ff5c454cc2daf9ec599b0752a4a118c8919af4679494b4667f5a0316f9f5'

package_bytes = base64.b64decode(BUNDLE_B64)
assert hashlib.sha256(package_bytes).hexdigest() == EXPECTED_BUNDLE_SHA256, 'Package checksum failed'
with zipfile.ZipFile(io.BytesIO(package_bytes)) as package:
    for item in package.infolist():
        if Path(item.filename).name != item.filename:
            raise ValueError('Unexpected path in package')
        destination = WORKDIR / item.filename
        data = package.read(item)
        if destination.exists() and destination.read_bytes() != data:
            raise ValueError('An existing prepared file differs: ' + item.filename + '. Keep your progress and ask for help before overwriting it.')
        if not destination.exists():
            destination.write_bytes(data)
spec = importlib.util.spec_from_file_location('movie1_uploader', WORKDIR / 'uploader.py')
uploader = importlib.util.module_from_spec(spec)
sys.modules['movie1_uploader'] = uploader
spec.loader.exec_module(uploader)
MESSAGES = json.loads((WORKDIR / 'result.cleaned.json').read_text(encoding='utf-8'))['messages']
MANIFEST = json.loads((WORKDIR / 'selected_photos.json').read_text(encoding='utf-8'))
SUMMARY = json.loads((WORKDIR / 'summary.json').read_text(encoding='utf-8'))
uploader.validate_plan(MESSAGES, MANIFEST)
print('Ready:', len(MESSAGES), 'posts. No messages have been sent.')
print('Files and progress folder:', WORKDIR)
print('Photo data to fetch: approximately', round(SUMMARY['selected_photo_bytes'] / 1_000_000), 'MB')


## 2. Review the cleanup

This displays only titles and metadata. The CSV and excluded-record file explain
every decision. The original GitHub history is the source, and the prepared copy
is separate. The same poster was used for two different films; both records remain.
No fuzzy matching deletes vaguely similar movie names or different episodes.


In [ ]:
#@title Show the selection summary and first 10 selected posts
print(json.dumps(SUMMARY, ensure_ascii=False, indent=2))
print('\nFirst 10 selected posts:')
for message in MESSAGES[:10]:
    print(message['id'], message['date'][:10], uploader.flattened(message).splitlines()[0])
print('\nTo review all decisions, open selection_audit.csv in the prepared Drive folder.')


In [ ]:
#@title Optional: inspect one complete saved caption (no Telegram sends)
SOURCE_ID_TO_PREVIEW = MESSAGES[0]['id']
message = next(m for m in MESSAGES if m['id'] == SOURCE_ID_TO_PREVIEW)
print('Source ID:', message['id'], '| Photo:', message.get('photo', 'text only'))
print(uploader.flattened(message))


## 3. Create the bot and connect the NEW channel

1. Create your new channel in Telegram.
2. Open [@BotFather](https://t.me/BotFather), send **/newbot**, and follow its prompts.
3. Add the resulting bot as an administrator of the new channel, allowing **Post Messages**.
4. Run the next cell. Enter the token in the hidden prompt, or save it in Colab Secrets
   under **TELEGRAM_BOT_TOKEN** and allow notebook access.
5. Enter the new channel's **@username** or numeric **-100... ID**. If the channel is
   private, leave the destination blank to list channels from the bot's recent updates.
   If the list is empty, post `MIGRATION_SETUP` manually in your new channel and rerun.

The connection cell reads account/channel information; it does not send a post.


In [ ]:
#@title Connect the dedicated bot (token hidden)
import getpass
token = None
try:
    from google.colab import userdata
    token = userdata.get('TELEGRAM_BOT_TOKEN')
except Exception:
    pass
if not token:
    token = getpass.getpass('Bot token (hidden): ').strip()
API = uploader.TelegramAPI(token)
del token
target = input('New channel @username or -100... ID; leave blank to list: ').strip()
if not target:
    channels = uploader.discover_channels(API)
    for channel in channels:
        print(channel['id'], channel.get('title', ''))
    if not channels:
        raise ValueError('No recent channel update. Add the bot as admin, post MIGRATION_SETUP in the new channel, and rerun this cell.')
    target = input('Enter the numeric ID of your intended NEW channel: ').strip()
if target.lstrip('-').isdigit():
    target = int(target)
BOT, CHANNEL = uploader.validate_target(API, target)
if CHANNEL['id'] == -1001881963160:
    raise ValueError('Select the NEW channel, not the original source channel.')
STATE_PATH = WORKDIR / ('progress_' + str(CHANNEL['id']) + '.json')
CACHE = WORKDIR / 'photo_cache'
state = uploader.load_state(STATE_PATH, MESSAGES, BOT['id'], CHANNEL['id'])
print('Destination:', CHANNEL['title'], '| ID:', CHANNEL['id'])
print('Bot:', BOT.get('username'), '| Already completed:', len(state['sent']))
if state.get('pending'):
    print('Pending delivery requires the recovery cell:', state['pending'])
print('Connection verified. No messages sent by this cell.')


## 4. Send FIVE posts first

This is the live integration check. It sends the next five selected posts to the
displayed channel. Check photos, captions, clickable links, and formatting in Telegram.
Each photo is one post with its complete saved caption. Text posts stay text.
After a test, use the continue cell; running this cell again sends the **next** five.


In [ ]:
#@title Send the next 5 posts — requires destination confirmation
expected = 'SEND 5 TO ' + CHANNEL['title']
answer = input('To post five messages, type exactly: ' + expected + '\n')
if answer == expected:
    state = uploader.run_upload(API, MESSAGES, MANIFEST, STATE_PATH, CACHE,
                                BOT['id'], CHANNEL['id'], limit=5, interval=3.0)
else:
    print('No posts sent.')


## 5. Continue all remaining posts

After checking the first five posts, run this cell. It resumes from successful
message IDs saved in Drive. Keep the computer/browser connected during the Colab run;
free runtime availability is not guaranteed. Do not run another uploader simultaneously.

The programmed pauses alone total about **2 hours 9 minutes** for the full plan,
plus photo transfers and Telegram-requested waits. This is not a completion-time promise.
Stop with the cell's stop button if needed. A send interrupted at an uncertain point
requires checking the last Telegram post before resuming.


In [ ]:
#@title Continue from saved progress — requires confirmation
expected = 'CONTINUE TO ' + CHANNEL['title']
answer = input('To upload all remaining posts, type exactly: ' + expected + '\n')
if answer == expected:
    state = uploader.run_upload(API, MESSAGES, MANIFEST, STATE_PATH, CACHE,
                                BOT['id'], CHANNEL['id'], limit=0, interval=3.0)
else:
    print('No posts sent.')


## Recovery — only if an upload stopped

Re-run preparation and connection with the **same bot, channel, and progress folder**.
Do not delete progress. A leftover lock can be cleared only after every previous upload
process has stopped. An uncertain send may already exist in Telegram: inspect it first.
The bot API does not provide a full history read to resolve ambiguous sends automatically.


In [ ]:
#@title Recover an interrupted send or stale lock (does not send messages)
lock = Path(str(STATE_PATH) + '.lock')
if lock.exists():
    confirmation = input('Only after stopping ALL other uploads, type ALL UPLOADS STOPPED to remove a stale lock: ')
    if confirmation != 'ALL UPLOADS STOPPED':
        raise ValueError('Lock retained; no changes made.')
    lock.unlink()
if STATE_PATH.exists():
    state = json.loads(STATE_PATH.read_text(encoding='utf-8'))
    pending = state.get('pending')
    if pending:
        print('Inspect this pending post in Telegram:', pending)
        action = input('After checking Telegram, type sent or not_sent (anything else stops): ').strip()
        new_id = None
        if action == 'sent':
            new_id = int(input('Actual NEW Telegram message ID (last number in copied post link): ').strip())
        if action not in ('sent', 'not_sent'):
            raise ValueError('No delivery status changed.')
        uploader.resolve_pending(STATE_PATH, pending['source_id'], action, new_id)
        print('Recovery recorded. You can use the continue cell.')
    else:
        print('No pending send. You can use the continue cell.')
else:
    print('No progress file exists yet.')


## Optional: download the prepared package for local use

The ZIP includes the cleaned JSON, all excluded records, selection audit, instructions,
and Python uploader. It excludes bot credentials and the large photo cache. If you have
already started, it includes progress JSON for resuming on the same destination.
On a PC with Python 3.10+, unzip it and run `python local_start.py`; no Codex is required.
Never move a job while the current uploader is still running.


In [ ]:
#@title Download package and current progress (no Telegram sends)
package_path = WORKDIR / 'Telegram_Movie1_Migration.zip'
base_names = ['result.cleaned.json', 'excluded_posts.json', 'selection_audit.csv',
              'selected_photos.json', 'summary.json', 'uploader.py', 'local_start.py',
              'prepare_migration.py', 'test_uploader.py', 'START_HERE.md']
progress_files = sorted(WORKDIR.glob('progress_*.json'))
with zipfile.ZipFile(package_path, 'w', zipfile.ZIP_DEFLATED) as package:
    for name in base_names:
        package.write(WORKDIR / name, arcname=name)
    for path in progress_files:
        package.write(path, arcname=path.name)
try:
    from google.colab import files
    files.download(str(package_path))
except ImportError:
    print('Package:', package_path)


## What has and has not been tested

All 2,573 selected records passed text/entity reconstruction, caption-length, photo
reference, and image-dimension validation. Ten mocked API tests passed for resume,
rate-limit waiting, uncertain-delivery handling, manual recovery, Unicode offsets,
wrong-destination/changed-plan checks, and concurrent-run prevention. Sample image bytes
and the source JSON hash were verified against the pinned GitHub snapshot.

No authenticated Telegram upload or end-to-end Colab execution has been performed.
The five-post step checks that integration using your new channel and your bot.
New posts receive new timestamps and IDs; subscribers, views, and reactions do not move.
The program uses Telegram's normal rate-limit responses; it does not bypass them.

References: [Telegram photo API](https://core.telegram.org/bots/api#sendphoto),
[Bot setup](https://core.telegram.org/bots/tutorial#obtain-your-bot-token),
[Telegram rate limits](https://core.telegram.org/bots/faq#my-bot-is-hitting-limits-how-do-i-avoid-this),
[Colab limitations](https://research.google.com/colaboratory/faq.html),
[Source snapshot](https://github.com/kawakaze430/Movie1/tree/3f8ff02f9b0a417205c317c36d2eaa73356955bf).
